<div dir="rtl">

# منصة تحويل الكتب العربية إلى Markdown

**الخطوة صفر — مهمة جدًا للكتب الكبيرة:** فعّل كرت الشاشة المجاني:
من القائمة **Runtime ← Change runtime type ← اختر T4 GPU ← Save**.
بدونه: كتاب 400 صفحة يأخذ ساعات. معه: نحو 15–30 دقيقة.

**ثم شغّل الخلايا بالترتيب** (زر ▶):

1. **التثبيت** — عدة دقائق، مرة واحدة لكل جلسة.
2. **ربط Drive** (اختياري) — نسخة احتياطية تلقائية للنتائج؛ تجاوزه إن أردت التنزيل المباشر فقط.
3. **تشغيل المنصة** — يطبع رابطًا: ارفع الكتاب، تابع التقدم، ثم **نزّل ZIP** واضغط **حذف من الخادم**.

⚠ اترك تبويب Colab مفتوحًا أثناء العمل؛ المعالجة تجري على خوادم Google لا على جهازك.

</div>

In [ ]:
#@title ١) التثبيت { display-mode: "form" }
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

try:
    subprocess.run(["nvidia-smi", "-L"], check=True, capture_output=True)
    print("✓ كرت شاشة GPU متاح — OCR سيعمل عليه (سريع)")
    pip("paddlepaddle-gpu==3.3.1", "-i",
        "https://www.paddlepaddle.org.cn/packages/stable/cu126/")
except Exception:
    print("⚠ لا يوجد GPU! سيعمل على المعالج (بطيء جدًا للكتب الكبيرة).")
    print("  فعّله: Runtime ← Change runtime type ← T4 GPU ← Save، ثم أعد تشغيل هذه الخلية.")
    pip("paddlepaddle==3.3.1")

pip("paddleocr[doc-parser]==3.7.0", "pymupdf", "fastapi", "uvicorn",
    "python-multipart", "markdown")
subprocess.run("rm -rf /content/book_ocr && git clone -q https://github.com/7aidaraa/book_ocr /content/book_ocr",
               shell=True, check=True)
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
               " -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
               shell=True, check=True)
print("✓ التثبيت اكتمل — شغّل الخلية التالية")

In [ ]:
#@title ٢) ربط Google Drive لحفظ النتائج (اختياري) { display-mode: "form" }
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

# نجعل مجلد النتائج داخل المنصة يشير إلى Drive مباشرة
target = Path("/content/drive/MyDrive/كتب-محوّلة")
target.mkdir(parents=True, exist_ok=True)

output_link = Path("/content/book_ocr/data/output")
output_link.parent.mkdir(parents=True, exist_ok=True)
if output_link.is_symlink() or output_link.exists():
    if output_link.is_symlink():
        output_link.unlink()
    else:
        import shutil; shutil.rmtree(output_link)
os.symlink(target, output_link)

print(f"\u2713 النتائج ستُحفظ في Drive داخل: كتب-محوّلة/")

In [ ]:
#@title ٣) تشغيل المنصة والحصول على الرابط { display-mode: "form" }
#@markdown رابط صفحة الوسيط الثابتة (Render) — تُسجَّل جلسة GPU هذه هناك تلقائيًا
#@markdown فيظهر زر «الدخول إلى المنصة السريعة» في صفحتك الثابتة:
render_hub = "https://book-ocr-n0f0.onrender.com"  #@param {type:"string"}
hub_token = "kitab-hub"  #@param {type:"string"}

import json, os, re, subprocess, sys, threading, time, urllib.request
from pathlib import Path

os.chdir("/content/book_ocr")

server = subprocess.Popen(
    [sys.executable, "run.py"],
    env={**os.environ, "HOST": "0.0.0.0", "PORT": "8000"},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

print("تشغيل الخادم...")
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/", timeout=2)
        break
    except Exception:
        if server.poll() is not None:
            raise SystemExit("✗ فشل تشغيل الخادم:\n" + server.stdout.read())
        time.sleep(1)
else:
    raise SystemExit("✗ الخادم لم يستجب")

print("فتح الرابط العام...")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
public_url = None
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line and tunnel.poll() is not None:
        break
    match = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise SystemExit("✗ تعذر إنشاء الرابط — أعد تشغيل هذه الخلية")

# تسجيل الجلسة لدى صفحة الوسيط + نبضة كل دقيقة لتبقى «شغّالة الآن»
def _register_forever():
    endpoint = render_hub.rstrip("/") + "/api/gpu-session"
    body = json.dumps({"url": public_url, "token": hub_token}).encode()
    while True:
        try:
            req = urllib.request.Request(
                endpoint, data=body, headers={"Content-Type": "application/json"})
            urllib.request.urlopen(req, timeout=30)
        except Exception:
            pass  # الوسيط نائم أو الشبكة متقطعة — سنعيد المحاولة
        time.sleep(60)

if render_hub.strip():
    threading.Thread(target=_register_forever, daemon=True).start()

print("\n" + "=" * 52)
print("  ✓ المنصة تعمل — افتح هذا الرابط:")
print(f"  {public_url}")
print("=" * 52)
if render_hub.strip():
    print(f"\n✓ سُجّلت الجلسة لدى الوسيط — أو ادخل دائمًا من:\n  {render_hub}")
print("\n⚠ اترك هذه الخلية تعمل ولا تغلق التبويب.")
print("   لإيقاف المنصة: اضغط زر التوقف ■ في هذه الخلية.\n")
print("--- سجل الخادم ---")
for line in server.stdout:
    print(line, end="")